# Token Cache Generation

This notebook creates the token caches used by the dynamic-graph forecasting model.

The canonical rolling-token files are stored by the production split:

- `encoded_data.pt`: January–August full-1,024 rolling tokens.
- `encoded_data_val.pt`: September full-1,024 rolling tokens.

For remapping, the notebook slices these canonical caches **in memory**:

- January–June: fit the retained top-\(K\) `s1` vocabulary.
- July–September: apply the fixed mapping and continue causal LOCF history.

It does **not** create or require duplicate `encoded_kronos_train.pt`,
`encoded_kronos_test.pt`, `decoded_kronos_train.pt`, or
`decoded_kronos_test.pt` files.

The model is trained from the origin-aligned caches, not the rolling caches.

---

## Context length 60, remapped `s1` vocabulary 150

### Existing canonical files reused

| File | Period | Required? |
|---|---|---|
| `encoded_data.pt` | January–August full-1,024 rolling tokens | Yes |
| `encoded_data_val.pt` | September full-1,024 rolling tokens | Yes |
| `origin_aligned_train_tokens.pt` | January–August full-1,024 model cache | Yes |
| `origin_aligned_val_tokens.pt` | September full-1,024 model cache | Yes |

### New required outputs

| File | Period | Required? |
|---|---|---|
| `s1_top150_c60_continuous_locf_mapping.pt` | Mapping fitted on January–June | Yes |
| `s1_top150_c60_continuous_locf_mapping.json` | Readable mapping summary | Yes, but not used directly by training |
| `origin_aligned_train_tokens_c60_s1_top150_locf.pt` | January–August compact model cache | Yes |
| `origin_aligned_val_tokens_c60_s1_top150_locf.pt` | September compact model cache | Yes |

### Optional outputs

Saved only when `SAVE_REMAPPED_ROLLING_CACHES=True`.

| File | Period | Required? |
|---|---|---|
| `encoded_kronos_train_c60_s1_top150_locf.pt` | January–June remapped rolling tokens | No |
| `encoded_kronos_jul_sep_c60_s1_top150_locf.pt` | July–September remapped rolling tokens | No |

---

## Context length 120, remapped `s1` vocabulary 150

### New canonical full-vocabulary outputs

| File | Period | Required? |
|---|---|---|
| `encoded_data_c120.pt` | January–August full-1,024 rolling tokens | Yes |
| `encoded_data_val_c120.pt` | September full-1,024 rolling tokens | Yes |
| `origin_aligned_train_tokens_c120.pt` | January–August full-1,024 model cache | Yes |
| `origin_aligned_val_tokens_c120.pt` | September full-1,024 model cache | Yes |

### New required compact outputs

| File | Period | Required? |
|---|---|---|
| `s1_top150_c120_continuous_locf_mapping.pt` | Mapping fitted on January–June 120-minute tokens | Yes |
| `s1_top150_c120_continuous_locf_mapping.json` | Readable mapping summary | Yes, but not used directly by training |
| `origin_aligned_train_tokens_c120_s1_top150_locf.pt` | January–August compact model cache | Yes |
| `origin_aligned_val_tokens_c120_s1_top150_locf.pt` | September compact model cache | Yes |

### Optional outputs

Saved only when `SAVE_REMAPPED_ROLLING_CACHES=True`.

| File | Period | Required? |
|---|---|---|
| `encoded_kronos_train_c120_s1_top150_locf.pt` | January–June remapped rolling tokens | No |
| `encoded_kronos_jul_sep_c120_s1_top150_locf.pt` | July–September remapped rolling tokens | No |

Changing the context length requires new full-vocabulary tokens because both the
context-only normalisation frame and the tokenizer's causal prefix change.


In [9]:
# All imports live in this cell.

from copy import deepcopy
import gc
from pathlib import Path
import sys
from time import perf_counter

import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.data_generator import WindowedCandleDataset
from src.data.load_candle_data import clean_candle_splits, load_candle_splits
from src.data.s1_token_remapping import (
    build_compact_origin_aligned_cache,
    build_train_validation_remapped_rolling_caches,
    concatenate_rolling_caches,
    fit_top_k_s1_remapping_resource,
    save_mapping_json_summary,
    save_s1_remapping_resource,
)
from src.data.token_graph_dataset import (
    build_and_save_origin_aligned_token_cache,
    load_origin_aligned_token_cache,
    save_origin_aligned_token_cache,
    validate_origin_aligned_token_cache,
)
from src.evaluation.tokenizer_metrics import (
    build_kronos_cache_period_view,
)
from src.models.dynamic_graph.config import (
    build_dense_window_config,
    load_dynamic_graph_config,
)
from src.models.kronos_tokenizer import (
    KronosTokenizerAdapter,
    encode_causal_split,
)
from src.utils.config import load_yaml

print("Project root:", PROJECT_ROOT)


Project root: /Users/vishalruparelia/Desktop/Thesis/dynamic_graphs_thesis


## Configuration

For the current experiment, use:


`CONTEXT_LENGTH = 60`
`PREDICTION_LENGTH = 60`
`S1_REMAP_K = 150`


The existing 60-minute canonical caches will be reused. Intermediate remapped
rolling caches are not required for training and are disabled by default to save
Drive space.

For a future 120-minute experiment, set `CONTEXT_LENGTH=120` and
`GENERATE_MISSING_ROLLING_CACHES=True`. The notebook will create new canonical
January–August and September rolling caches, then slice the remapping periods in
memory.


In [10]:
# Core experiment settings.
CONTEXT_LENGTH = 60
PREDICTION_LENGTH = 60
S1_REMAP_K = 250       # Set to None to build only the full 1024-token caches.
EVALUATION_HORIZONS = (1, 5, 15, 30, 60)

# Encoding throughput.
WINDOW_BATCH_SIZE = 2
ROLLING_WINDOW_BATCH_SIZE = 8
SERIES_BATCH_SIZE = 93

# File behaviour.
GENERATE_MISSING_ROLLING_CACHES = False
OVERWRITE_FULL_ORIGIN_CACHES = False
OVERWRITE_COMPACT_CACHES = False

# Intermediate remapped rolling caches are not used for training.
# Keep False unless a later analysis explicitly needs them.
SAVE_REMAPPED_ROLLING_CACHES = False

RUN_DECODE_SMOKE_TEST = True
DECODE_SMOKE_WINDOWS = 2

DATA_DIR = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "Shared drives/Vishal/data/cached_datasets/"
    "exp-1m-95s-24y/session"
)

TOKEN_DIR = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "My Drive/dissertation/final_model/tokens"
)

FORECASTING_CONFIG_PATH = PROJECT_ROOT / "configs" / "forecasting.yaml"
DYNAMIC_GRAPH_CONFIG_PATH = PROJECT_ROOT / "configs" / "dynamic_graph.yaml"

# Canonical full-vocabulary rolling caches are stored by production split:
# January--August training and September validation.
if CONTEXT_LENGTH == 60:
    FULL_ROLLING_TRAIN_PATH = TOKEN_DIR / "encoded_data.pt"
    FULL_ROLLING_VAL_PATH = TOKEN_DIR / "encoded_data_val.pt"

    FULL_ORIGIN_TRAIN_PATH = TOKEN_DIR / "origin_aligned_train_tokens.pt"
    FULL_ORIGIN_VAL_PATH = TOKEN_DIR / "origin_aligned_val_tokens.pt"
else:
    FULL_ROLLING_TRAIN_PATH = TOKEN_DIR / (
        f"encoded_data_c{CONTEXT_LENGTH}.pt"
    )
    FULL_ROLLING_VAL_PATH = TOKEN_DIR / (
        f"encoded_data_val_c{CONTEXT_LENGTH}.pt"
    )

    FULL_ORIGIN_TRAIN_PATH = TOKEN_DIR / (
        f"origin_aligned_train_tokens_c{CONTEXT_LENGTH}.pt"
    )
    FULL_ORIGIN_VAL_PATH = TOKEN_DIR / (
        f"origin_aligned_val_tokens_c{CONTEXT_LENGTH}.pt"
    )

if S1_REMAP_K is not None:
    REMAP_RESOURCE_PATH = TOKEN_DIR / (
        f"s1_top{S1_REMAP_K}_c{CONTEXT_LENGTH}_continuous_locf_mapping.pt"
    )
    REMAP_RESOURCE_JSON_PATH = REMAP_RESOURCE_PATH.with_suffix(".json")

    # Optional intermediate period views.
    REMAPPED_ROLLING_TRAIN_PATH = TOKEN_DIR / (
        f"encoded_kronos_train_c{CONTEXT_LENGTH}_"
        f"s1_top{S1_REMAP_K}_locf.pt"
    )
    REMAPPED_ROLLING_LATER_PATH = TOKEN_DIR / (
        f"encoded_kronos_jul_sep_c{CONTEXT_LENGTH}_"
        f"s1_top{S1_REMAP_K}_locf.pt"
    )

    # Required model-training caches.
    COMPACT_ORIGIN_TRAIN_PATH = TOKEN_DIR / (
        f"origin_aligned_train_tokens_c{CONTEXT_LENGTH}_"
        f"s1_top{S1_REMAP_K}_locf.pt"
    )
    COMPACT_ORIGIN_VAL_PATH = TOKEN_DIR / (
        f"origin_aligned_val_tokens_c{CONTEXT_LENGTH}_"
        f"s1_top{S1_REMAP_K}_locf.pt"
    )

TOKEN_DIR.mkdir(parents=True, exist_ok=True)

for required in (
    DATA_DIR,
    FORECASTING_CONFIG_PATH,
    DYNAMIC_GRAPH_CONFIG_PATH,
):
    if not required.exists():
        raise FileNotFoundError(required)

print("Context length:", CONTEXT_LENGTH)
print("Prediction length:", PREDICTION_LENGTH)
print("Compact s1 vocabulary:", S1_REMAP_K)
print("Token directory:", TOKEN_DIR)
print("Canonical rolling train:", FULL_ROLLING_TRAIN_PATH)
print("Canonical rolling validation:", FULL_ROLLING_VAL_PATH)


Context length: 60
Prediction length: 60
Compact s1 vocabulary: 250
Token directory: /Users/vishalruparelia/Library/CloudStorage/GoogleDrive-vishal@autonomous-fox.ai/My Drive/dissertation/final_model/tokens
Canonical rolling train: /Users/vishalruparelia/Library/CloudStorage/GoogleDrive-vishal@autonomous-fox.ai/My Drive/dissertation/final_model/tokens/encoded_data.pt
Canonical rolling validation: /Users/vishalruparelia/Library/CloudStorage/GoogleDrive-vishal@autonomous-fox.ai/My Drive/dissertation/final_model/tokens/encoded_data_val.pt


## Load production data, configuration, and frozen tokenizer

The raw candle loader supplies the canonical production periods:

- training: January–August;
- validation: September.

The frozen tokenizer is required only when a missing full cache must be generated
or when the final compact-ID decoder smoke test is enabled.

The remapping fit and later-history periods are derived from the canonical rolling
caches in the next section; they are not saved as duplicate full-vocabulary files.


In [11]:
forecasting_config = load_yaml(
    FORECASTING_CONFIG_PATH
)

experiment_config = load_dynamic_graph_config(
    DYNAMIC_GRAPH_CONFIG_PATH,
    preset="structured_parallel_uniform",
)
experiment_config = deepcopy(experiment_config)

experiment_config["forecasting"]["context_length"] = CONTEXT_LENGTH
experiment_config["forecasting"]["prediction_length"] = PREDICTION_LENGTH
experiment_config["models"]["dynamic_graph"]["context_length"] = CONTEXT_LENGTH
experiment_config["models"]["dynamic_graph"]["heads"][
    "prediction_length"
] = PREDICTION_LENGTH

train_raw, val_raw, test_raw = load_candle_splits(
    DATA_DIR
)
train, val, test = clean_candle_splits(
    train_raw,
    val_raw,
    test_raw,
)

kronos_tokenizer = (
    KronosTokenizerAdapter.from_config(
        forecasting_config,
        series_batch_size=SERIES_BATCH_SIZE,
    )
    .load()
)

print("Production train sessions (Jan--Aug):", len(train["samples"]))
print("Production validation sessions (Sep):", len(val["samples"]))
print("Tokenizer:", kronos_tokenizer.tokenizer_id)


Production train sessions (Jan--Aug): 167
Production validation sessions (Sep): 20
Tokenizer: NeoQuasar/Kronos-Tokenizer-base


## Load or generate canonical full-vocabulary rolling caches

Only two canonical rolling caches are stored for each context length:

- the full January–August production training period;
- the September production validation period.

For remapping, this section creates temporary in-memory views:

- January–June, used to fit the retained top-\(K\) IDs;
- July–September, used to continue the fixed remapping and LOCF history.

No duplicate `encoded_kronos_train.pt` or `encoded_kronos_test.pt` files are
created or required.


In [12]:
def atomic_torch_save(value, path):
    path = Path(path).expanduser().resolve()
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_name(
        f".{path.name}.tmp"
    )
    if temporary.exists():
        temporary.unlink()

    torch.save(value, temporary)
    temporary.replace(path)
    return path


def load_or_generate_full_rolling(
    split,
    path,
    *,
    label,
):
    path = Path(path).expanduser().resolve()

    if path.is_file():
        print(f"Loading {label} canonical rolling cache:", path)
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )

    if not GENERATE_MISSING_ROLLING_CACHES:
        raise FileNotFoundError(
            f"Missing {label} canonical rolling cache: {path}. "
            "Set GENERATE_MISSING_ROLLING_CACHES=True to create it."
        )

    print(f"Generating {label} canonical rolling cache:", path)
    start = perf_counter()

    encoded = encode_causal_split(
        kronos_tokenizer,
        split,
        context_length=CONTEXT_LENGTH,
        window_batch_size=ROLLING_WINDOW_BATCH_SIZE,
        series_batch_size=SERIES_BATCH_SIZE,
        show_progress=True,
    )

    atomic_torch_save(
        encoded,
        path,
    )

    print(
        f"{label} rolling generation: "
        f"{(perf_counter() - start) / 60:.2f} min"
    )
    return encoded


full_rolling_train = load_or_generate_full_rolling(
    train,
    FULL_ROLLING_TRAIN_PATH,
    label="January--August",
)

full_rolling_val = load_or_generate_full_rolling(
    val,
    FULL_ROLLING_VAL_PATH,
    label="September",
)

# Create the two remapping periods in memory. Nothing is written here.
rolling_train = build_kronos_cache_period_view(
    (full_rolling_train,),
    start_date="2024-01-01",
    end_date="2024-07-01",
    name="remapping fit: January--June",
)

rolling_later = build_kronos_cache_period_view(
    (
        full_rolling_train,
        full_rolling_val,
    ),
    start_date="2024-07-01",
    end_date="2024-10-01",
    name="remapping continuation: July--September",
)

# Release the large canonical dictionaries after the period views exist.
del full_rolling_train
del full_rolling_val
gc.collect()

print("Remapping-fit sessions (Jan--Jun):", len(rolling_train["dates"]))
print("Later-history sessions (Jul--Sep):", len(rolling_later["dates"]))
print("Rolling fit context_s1:", tuple(rolling_train["context_s1"].shape))
print("Rolling later context_s1:", tuple(rolling_later["context_s1"].shape))
print(
    "Fit date range:",
    rolling_train["dates"][0],
    "to",
    rolling_train["dates"][-1],
)
print(
    "Later date range:",
    rolling_later["dates"][0],
    "to",
    rolling_later["dates"][-1],
)


Loading January--August canonical rolling cache: /Users/vishalruparelia/Library/CloudStorage/GoogleDrive-vishal@autonomous-fox.ai/My Drive/dissertation/final_model/tokens/encoded_data.pt
Loading September canonical rolling cache: /Users/vishalruparelia/Library/CloudStorage/GoogleDrive-vishal@autonomous-fox.ai/My Drive/dissertation/final_model/tokens/encoded_data_val.pt
Remapping-fit sessions (Jan--Jun): 124
Later-history sessions (Jul--Sep): 63
Rolling fit context_s1: (124, 331, 60, 93)
Rolling later context_s1: (63, 331, 60, 93)
Fit date range: 2024-01-02 to 2024-06-28
Later date range: 2024-07-01 to 2024-09-30


## Load or generate full-vocabulary origin-aligned caches

These are the model-ready full-1,024 caches. Each item contains the observed
context and the dense 60-minute future token path aligned to one forecasting
origin.

Normalisation statistics are calculated from the observed context only and then
held fixed for the complete context-plus-future tokenisation path.

For `CONTEXT_LENGTH=60`, the existing canonical files are reused. A new context
length generates new files with the context length in the filename.


In [13]:
dense_window_config = build_dense_window_config(
    forecasting_config,
    experiment_config,
)

dense_train_dataset = WindowedCandleDataset.from_config(
    split=train,
    config=dense_window_config,
    normaliser=None,
)
dense_val_dataset = WindowedCandleDataset.from_config(
    split=val,
    config=dense_window_config,
    normaliser=None,
)

expected_horizons = tuple(range(1, PREDICTION_LENGTH + 1))
for label, dataset in (("train", dense_train_dataset), ("validation", dense_val_dataset)):
    if tuple(dataset.horizons) != expected_horizons:
        raise RuntimeError(f"{label} dataset does not use dense horizons 1...P.")
    if dataset.context_length != CONTEXT_LENGTH:
        raise RuntimeError(f"{label} context length is incorrect.")


def load_or_generate_origin(dataset, path, *, label):
    path = Path(path).expanduser().resolve()
    if path.is_file() and not OVERWRITE_FULL_ORIGIN_CACHES:
        print(f"Loading {label} full origin cache:", path)
        return load_origin_aligned_token_cache(path)

    if path.exists():
        path.unlink()

    print(f"Generating {label} full origin cache:", path)
    start = perf_counter()
    build_and_save_origin_aligned_token_cache(
        dataset=dataset,
        tokenizer=kronos_tokenizer,
        path=path,
        evaluation_horizons=EVALUATION_HORIZONS,
        window_batch_size=WINDOW_BATCH_SIZE,
        series_batch_size=SERIES_BATCH_SIZE,
        prefix_check_batches=1,
        show_progress=True,
    )
    print(f"{label} origin generation: {(perf_counter() - start) / 60:.2f} min")
    return load_origin_aligned_token_cache(path)

full_origin_train = load_or_generate_origin(
    dense_train_dataset,
    FULL_ORIGIN_TRAIN_PATH,
    label="training",
)
full_origin_val = load_or_generate_origin(
    dense_val_dataset,
    FULL_ORIGIN_VAL_PATH,
    label="validation",
)

print("Full origin train context:", tuple(full_origin_train["context_tokens"].shape))
print("Full origin validation context:", tuple(full_origin_val["context_tokens"].shape))

Loading training full origin cache: /Users/vishalruparelia/Library/CloudStorage/GoogleDrive-vishal@autonomous-fox.ai/My Drive/dissertation/final_model/tokens/origin_aligned_train_tokens.pt
Loading validation full origin cache: /Users/vishalruparelia/Library/CloudStorage/GoogleDrive-vishal@autonomous-fox.ai/My Drive/dissertation/final_model/tokens/origin_aligned_val_tokens.pt
Full origin train context: (3173, 60, 93, 2)
Full origin validation context: (380, 60, 93, 2)


## Fit the compact vocabulary and apply continuous causal LOCF

The retained original Kronos IDs are selected from January–June token
frequencies only. The same mapping is then applied chronologically through
July–September.

A discarded token is replaced by the most recent earlier original token that
belongs to the retained set. The carry continues across context and session
boundaries. The least frequent retained token is used only if an asset has no
retained token anywhere in its available earlier history.

The full-vocabulary period views exist only in memory. Optional remapped rolling
views are saved only when `SAVE_REMAPPED_ROLLING_CACHES=True`.


In [14]:
if S1_REMAP_K is None:
    print("S1_REMAP_K=None: compact-cache generation is disabled.")
else:
    remap_resource = fit_top_k_s1_remapping_resource(
        rolling_train,
        k=S1_REMAP_K,
    )
    save_s1_remapping_resource(remap_resource, REMAP_RESOURCE_PATH)
    save_mapping_json_summary(remap_resource, REMAP_RESOURCE_JSON_PATH)

    remapped_rolling_train, remapped_rolling_later, fallback_assets = (
        build_train_validation_remapped_rolling_caches(
            rolling_train,
            rolling_later,
            remap_resource,
        )
    )

    if SAVE_REMAPPED_ROLLING_CACHES:
        atomic_torch_save(remapped_rolling_train, REMAPPED_ROLLING_TRAIN_PATH)
        atomic_torch_save(remapped_rolling_later, REMAPPED_ROLLING_LATER_PATH)

    original_rolling_all = concatenate_rolling_caches(
        (rolling_train, rolling_later),
        name="original Jan--Sep",
    )
    remapped_rolling_all = concatenate_rolling_caches(
        (remapped_rolling_train, remapped_rolling_later),
        name="remapped Jan--Sep",
    )

    print("Retained original s1 IDs:", remap_resource.k)
    print("Training token coverage (%):", f"{remap_resource.training_coverage_percent:.4f}")
    print("Emergency fallback original ID:", remap_resource.fallback_original_id)
    print("Assets requiring emergency fallback:", fallback_assets)
    print("Mapping hash:", remap_resource.resource_hash)

Retained original s1 IDs: 250
Training token coverage (%): 98.8959
Emergency fallback original ID: 980
Assets requiring emergency fallback: ()
Mapping hash: b01b8fe6e3d803732e0e7e8929dd761c3b0672b9829a31b5f0b77929bcca238f


## Build compact origin-aligned train and validation caches

Observed context `s1` IDs are taken from the remapped rolling cache at the exact session date and forecast origin. Dense future labels are remapped causally from the last retained context token.

The model-facing cache stores compact IDs `0...K-1`. It also stores the exact compact-to-original lookup needed by the frozen decoder. `s2`, raw evaluation targets, context statistics, dates, origins, and asset ordering remain unchanged.

In [15]:
if S1_REMAP_K is not None:
    if (
        COMPACT_ORIGIN_TRAIN_PATH.exists()
        and COMPACT_ORIGIN_VAL_PATH.exists()
        and not OVERWRITE_COMPACT_CACHES
    ):
        print("Loading existing compact origin caches.")

        compact_origin_train = load_origin_aligned_token_cache(
            COMPACT_ORIGIN_TRAIN_PATH
        )
        compact_origin_val = load_origin_aligned_token_cache(
            COMPACT_ORIGIN_VAL_PATH
        )
    else:
        compact_origin_train = build_compact_origin_aligned_cache(
            full_origin_train,
            original_rolling_all,
            remapped_rolling_all,
            remap_resource,
            split_name="train",
        )

        compact_origin_val = build_compact_origin_aligned_cache(
            full_origin_val,
            original_rolling_all,
            remapped_rolling_all,
            remap_resource,
            split_name="validation",
        )

        save_origin_aligned_token_cache(
            compact_origin_train,
            COMPACT_ORIGIN_TRAIN_PATH,
        )
        save_origin_aligned_token_cache(
            compact_origin_val,
            COMPACT_ORIGIN_VAL_PATH,
        )

    validate_origin_aligned_token_cache(
        compact_origin_train
    )
    validate_origin_aligned_token_cache(
        compact_origin_val
    )

    for label, cache in (
        ("training", compact_origin_train),
        ("validation", compact_origin_val),
    ):
        if (
            cache["s1_remapping_resource_hash"]
            != remap_resource.resource_hash
        ):
            raise RuntimeError(
                f"Existing {label} compact cache was generated with "
                "a different remapping resource. Set "
                "OVERWRITE_COMPACT_CACHES=True to rebuild it."
            )

    if (
        compact_origin_train["s1_remapping_resource_hash"]
        != compact_origin_val["s1_remapping_resource_hash"]
    ):
        raise RuntimeError(
            "Train and validation compact caches use different mappings."
        )

    if (
        compact_origin_train["asset_cols"]
        != compact_origin_val["asset_cols"]
    ):
        raise RuntimeError(
            "Train and validation asset ordering differs."
        )

    print("Compact train cache:", COMPACT_ORIGIN_TRAIN_PATH)
    print("Compact validation cache:", COMPACT_ORIGIN_VAL_PATH)
    print(
        "Train context:",
        tuple(compact_origin_train["context_tokens"].shape),
    )
    print(
        "Validation context:",
        tuple(compact_origin_val["context_tokens"].shape),
    )
    print(
        "Train target_s1:",
        tuple(compact_origin_train["target_s1"].shape),
    )
    print(
        "Validation target_s1:",
        tuple(compact_origin_val["target_s1"].shape),
    )
    print(
        "Observed compact s1 range:",
        int(compact_origin_train["context_tokens"][..., 0].min()),
        "to",
        int(compact_origin_train["context_tokens"][..., 0].max()),
    )


Compact train cache: /Users/vishalruparelia/Library/CloudStorage/GoogleDrive-vishal@autonomous-fox.ai/My Drive/dissertation/final_model/tokens/origin_aligned_train_tokens_c60_s1_top250_locf.pt
Compact validation cache: /Users/vishalruparelia/Library/CloudStorage/GoogleDrive-vishal@autonomous-fox.ai/My Drive/dissertation/final_model/tokens/origin_aligned_val_tokens_c60_s1_top250_locf.pt
Train context: (3173, 60, 93, 2)
Validation context: (380, 60, 93, 2)
Train target_s1: (3173, 60, 93)
Validation target_s1: (380, 60, 93)
Observed compact s1 range: 0 to 249


## Decoder smoke test

This converts the compact IDs back to retained original Kronos IDs, decodes a small number of validation windows with the genuine frozen coarse decoder, and verifies finite output and expected shapes. This is a wiring test; the full reconstruction sweep remains the evidence used to choose `K`.

In [16]:
if S1_REMAP_K is not None and RUN_DECODE_SMOKE_TEST:
    count = min(DECODE_SMOKE_WINDOWS, compact_origin_val["context_tokens"].shape[0])
    inverse = torch.as_tensor(
        compact_origin_val["s1_compact_to_original"],
        dtype=torch.long,
    )

    context_compact = torch.as_tensor(
        compact_origin_val["context_tokens"][:count]
    ).to(torch.long)
    target_compact = torch.as_tensor(
        compact_origin_val["target_s1"][:count]
    ).to(torch.long)

    context_original = context_compact.clone()
    context_original[..., 0] = inverse[context_compact[..., 0]]
    target_original = inverse[target_compact]

    decoded = kronos_tokenizer.decode_coarse_token_path(
        context_original,
        target_original,
        mean=torch.as_tensor(compact_origin_val["context_mean"][:count]),
        std=torch.as_tensor(compact_origin_val["context_std"][:count]),
        series_batch_size=SERIES_BATCH_SIZE,
        return_full_path=False,
    ).to(torch.float32)

    expected_shape = (
        count,
        PREDICTION_LENGTH,
        len(compact_origin_val["asset_cols"]),
        5,
    )
    if tuple(decoded.shape) != expected_shape:
        raise RuntimeError(
            f"Decoded shape {tuple(decoded.shape)} != {expected_shape}."
        )
    if not torch.isfinite(decoded).all():
        raise RuntimeError("Decoded compact-token path contains non-finite values.")

    evaluation_indices = torch.tensor(
        [horizon - 1 for horizon in EVALUATION_HORIZONS],
        dtype=torch.long,
    )
    decoded_eval_close = decoded.index_select(1, evaluation_indices)[..., 3]
    true_eval_close = torch.as_tensor(
        compact_origin_val["evaluation_true"][:count]
    )[..., 3]
    close_error_bps = (
        torch.log(decoded_eval_close.clamp_min(1e-8) / true_eval_close.clamp_min(1e-8))
        .abs()
        .mul(10_000.0)
    )

    print("Decoder smoke test passed.")
    print("Decoded future shape:", tuple(decoded.shape))
    print("Selected-horizon median close error (bps):", float(close_error_bps.median()))

Decoder smoke test passed.
Decoded future shape: (2, 60, 93, 5)
Selected-horizon median close error (bps): 1.9492617845535278


## Training outputs and safe cleanup

For the current 60-minute, top-150 run, pass these files to the training runner:

```text
origin_aligned_train_tokens_c60_s1_top150_locf.pt
origin_aligned_val_tokens_c60_s1_top150_locf.pt
```

The model configuration must use:

```text
heads.s1_vocabulary_size = 150
heads.s2_vocabulary_size = 1024
future_token_mode = coarse_only
```

The training runner maps compact generated `s1` IDs back to their retained
original Kronos IDs before frozen coarse decoding. The mapping hash is stored in
the caches, run metadata, and resume signature.

After this notebook and the cleaned tokenizer-analysis notebook both run
successfully, these duplicate historical period files are no longer required and
can be deleted from Drive:

```text
encoded_kronos_train.pt
encoded_kronos_test.pt
decoded_kronos_train.pt
decoded_kronos_test.pt
```

Do not delete the canonical files:

```text
encoded_data.pt
encoded_data_val.pt
decoded_data.pt
decoded_data_val.pt
origin_aligned_train_tokens.pt
origin_aligned_val_tokens.pt
```
